# Structure 
At the moment, the progression I'm thinking of is to do increasingly complex examples of ice fractures.  This notebook will cover mostly theory.
The individual examples will be in different notebooks:
- **Step 1**: Sanity check, single crack
- **Step 2**: Single crack between two crack tips
- **Step 3**: 3 colinear crack tips
- **Step 4**: 3 crack tips forming an angle
- **Step 5**: 4 and 5 crack tips
- **Extension 1**: 3D fracture network, fracture faces and mode 3 
- **Extension 2**: realistic boundary and load conditions

# Problem
We are applying the paper by Recho to convert the fractures in continuum mechanics into  hamiltonian formalism. This changes the high-order PDEs into a first order ODEs. 
By solving, we get the eigenvalues and eigenvectors, which correspond to the fracture tip fields. 

# Recho Continuum Fracture Hamiltonian
We want to calculate the eigenvalues of $\displaystyle \frac{\partial \psi(\theta)}{\partial \theta} = H(\theta)\psi(\theta)$ for the following Hamiltonian.
$$H(\theta) = \begin{pmatrix}
E_1 -C_d^{-1}C_f\lambda & C_d^{-1} \\
E_3(C_d^{-1}C_f)\lambda^2 & E_1+(E_2+E_3C_d^{-1})\lambda
\end{pmatrix}$$

where $$E_1 = \begin{pmatrix}0 & -1 \\1 & 0 \end{pmatrix}, E_2 = \begin{pmatrix}0 & -1 \\0 & 0 \end{pmatrix}, E_3 = \begin{pmatrix}0 & 0 \\-1 & 0 \end{pmatrix}$$
and 
$C_d, C_f$ can be derived from the stress-strain relationship, $\sigma = C~\varepsilon$



Continuity conditions also must be met for angles of the V-notch fractures: $$\psi^{(1)}(\theta = \theta_1) = \psi^{(2)}(\theta = \theta_1), ..., \psi^{(n-1)}(\theta = \theta_{n-1}) = \psi^{(n)}(\theta = \theta_{n-1})$$

"Any numerical method providing a good accuracy can
be used for solving this problem and the eigenvectors $\psi$ can straightforwardly be given with all stress and
displacement components." - Recho

Assumption, we will model all cracks as straight, and as free traction cracks. Which means there is no force directly on the surface of the crack, only around it.  

# Fracture Mechanics Formalism of Recho Hamiltonian

$$\sigma = C~\varepsilon$$
$$\sigma = \{\sigma_r, \sigma_\theta, \tau_{r\theta}\}^T$$
$$\varepsilon = \{\varepsilon_r, \varepsilon_\theta, \gamma_{r\theta}\}^T$$


## Equilibrium Relations
$$\frac{\partial{\sigma_r}}{\partial{r}} = $$

## Displacement-Stress Relations

# Simplest Case for Hamiltonian problem

## Assumptions and simplifications
The fracture volume will be homogeneous and isotropic, i.e. all made of the same stuff.
This let's us keep the eigenvalue $\lambda$ real. also, 
$$\sigma_{ij}~ \tilde ~ r^{\lambda-1}$$


In [2]:
import numpy as np
import sympy as sp

## Setting up the problem
### Assumptions:
- 2D inifinite plane
- isotropic homogeneous ice, elastic
- young's modulus $E = 9 GPa$
- poisson's ratio $\nu = 0.3$
### Set up:
- the crack face is at $\theta = \pm \pi$
- set up polar coordinates around the crack tip as (r, $\theta$)
- We will solve the equation with the Recho Hamiltonian to find the stress and displacement:
$$\sigma_{ij}(r,\theta),~~ u(r,\theta)$$


In [4]:
E_ice = 9e9      # Pa
nu_ice = 0.30    # Poisson's ratio

$$\sigma = C~\varepsilon$$
$$\sigma = \{\sigma_r, \sigma_\theta, \tau_{r\theta}\}^T$$
$$\varepsilon = \{\varepsilon_r, \varepsilon_\theta, \gamma_{r\theta}\}^T$$
This in matrix form:
$$\begin{bmatrix}
\sigma_r\\
\sigma_\theta\\
\tau_{r\theta}
\end{bmatrix} = \begin{bmatrix}
C_{11}\ C_{12}\ ~0~\\
C_{21}\ C_{22}\ ~0~\\
~0~~\ ~~0~~\ C_{33}
\end{bmatrix}\begin{bmatrix}
\varepsilon_r\\ \varepsilon_\theta\\ \gamma_{r\theta}
\end{bmatrix}
$$


In [7]:
# Plane strain stiffnesses (Voigt: [ε_rr, ε_θθ, γ_rθ])
def stiffness_plane_strain(E, nu):
    """
    Return 3x3 stiffness matrix C for plane strain in polar Voigt form:
    [σ_rr, σ_θθ, τ_rθ]^T = C [ε_rr, ε_θθ, γ_rθ]^T
    """
    factor = E / ((1 + nu) * (1 - 2 * nu))
    C11 = factor * (1 - nu)
    C12 = factor * nu
    C33 = E / (2 * (1 + nu))  # shear modulus * 2

    C = np.array([
        [C11, C12, 0.0],
        [C12, C11, 0.0],
        [0.0,  0.0, C33]
    ], dtype=float)
    return C

C = stiffness_plane_strain(E_ice, nu_ice)
print("Stiffness matrix C (Pa):")
print(C)

Stiffness matrix C (Pa):
[[1.21153846e+10 5.19230769e+09 0.00000000e+00]
 [5.19230769e+09 1.21153846e+10 0.00000000e+00]
 [0.00000000e+00 0.00000000e+00 3.46153846e+09]]


## Recho Hamiltonian Set up
These are the vectors we setup:

stress:
$$\sigma = 
\begin{bmatrix}
\sigma_{rr}\\
\sigma_{\theta\theta}\\
\sigma_{r\theta}
\end{bmatrix}
$$
strain:
$$\varepsilon = 
\begin{bmatrix}
\varepsilon_r\\ 
\varepsilon_\theta\\ 
\gamma_{r\theta}
\end{bmatrix}
$$
displacement:
$$q = 
\begin{bmatrix}
u_r\\
u_\theta
\end{bmatrix}
$$
We are going to do a change of variables in line with the paper: $\xi = ln(r)$ 

Which means $r = \exp(\xi),~~~ \frac{\partial{}}{\partial{r}} = \frac{1}{r}\frac{\partial{r}}{\partial{\xi}}$

We also need to switch the stress vector with the generalised version used by Recho:
$$p = 
\begin{bmatrix}
S_r\\
S_\theta
\end{bmatrix}=
\begin{bmatrix}
r\sigma_{rr}\\
r\sigma_{r\theta}
\end{bmatrix}$$

## State Vector
Following the paper, we get the following:
$$\dot{q} = H_{11}q + H_{12}p$$
$$\dot{p} = H_{21}q + H_{22}p$$

And we can define a state vector $v(\xi, \theta)$:
$$v(\xi, \theta) = \begin{bmatrix}q(\xi, \theta)\\p(\xi, \theta)\end{bmatrix} = \begin{bmatrix}S_r\\S_\theta\\u_r\\u_\theta\end{bmatrix}$$

So,
$$\dot{v} = H v$$


## Separation of variables


We will first look at a simple fracture with 'nice' angles. 




In [1]:
import qiskit
import math
import rustworkx as rx
from rustworkx.visualization import mpl_draw
import matplotlib.pyplot as plt

## Single Crack
We will first formulate for a single crack across.  

### Define the crack


### Show graph of fracture

### formulate hamiltonian

### solve hamiltonian